
### **混淆矩阵与错误样本分析 | TensorBoard 可视化 | 指标评估与可视化工具**

---

#### **一、混淆矩阵（Confusion Matrix）概述**

##### 1.1 **定义**

混淆矩阵是一个用于评估分类模型性能的工具。它以矩阵的形式展示模型的预测结果与实际标签之间的关系。

* 混淆矩阵是一个 **二维矩阵**，用于描述分类模型在分类问题中的性能。
* 在二分类问题中，矩阵通常是 2x2 的形式，如下所示：

| Actual \ Predicted | 0 (Negative)        | 1 (Positive)        |
| ------------------ | ------------------- | ------------------- |
| **0 (Negative)**   | True Negative (TN)  | False Positive (FP) |
| **1 (Positive)**   | False Negative (FN) | True Positive (TP)  |

##### 1.2 **各项元素解释**

* **True Positive (TP)**：实际为正类，且被预测为正类。
* **False Positive (FP)**：实际为负类，且被预测为正类（误判为正）。
* **False Negative (FN)**：实际为正类，且被预测为负类（误判为负）。
* **True Negative (TN)**：实际为负类，且被预测为负类。

##### 1.3 **混淆矩阵的作用**

* 评估分类模型的性能。
* 帮助识别模型的误分类类型（如误将正类预测为负类等）。
* 提供了更细粒度的错误分析，帮助开发者改进模型。

---

#### **二、基于混淆矩阵的性能度量**

##### 2.1 **常用评估指标**

从混淆矩阵中可以计算出一些常用的模型评估指标。

* **准确率（Accuracy）**：表示模型正确预测的比例。

  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$

* **精确率（Precision）**：表示模型预测为正类时，实际为正类的比例。

  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

* **召回率（Recall）**：表示实际为正类时，模型预测为正类的比例。

  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

* **F1分数（F1-Score）**：精确率和召回率的调和平均值。

  $$
  F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
  $$

##### 2.2 **性能评估与模型改进**

* **准确率**高不代表模型好，尤其是数据集不均衡时。
* **精确率**适用于正类预测的准确性要求高的场景。
* **召回率**适用于希望尽可能捕捉所有正类的场景，如医学检测。
* **F1分数**综合考虑精确率和召回率，是一个更全面的指标。

---

#### **三、错误样本分析**

##### 3.1 **为什么进行错误样本分析**

通过错误样本分析，我们可以发现模型的弱点，进一步改进模型的性能。

* **False Positive（误判为正）**：例如，垃圾邮件分类器错误地将正常邮件分类为垃圾邮件。
* **False Negative（误判为负）**：例如，肿瘤检测模型错误地将癌症患者判断为健康。

##### 3.2 **通过混淆矩阵进行错误样本分析**

* 可以通过混淆矩阵中的 **FP** 和 **FN** 定位错误样本。
* 分析这些样本的特征和模型误判的原因，有助于优化特征选择、改进模型结构等。

---

#### **四、TensorBoard 可视化**

##### 4.1 **TensorBoard简介**

TensorBoard 是 TensorFlow 提供的一款可视化工具，用于查看和分析机器学习实验的日志。它提供了多种可视化功能，如曲线、图像、文本等，帮助开发者监控模型的训练过程。

##### 4.2 **如何使用 TensorBoard**

1. **初始化 SummaryWriter**：

   ```python
   from torch.utils.tensorboard import SummaryWriter
   writer = SummaryWriter(log_dir='runs/experiment_1')
   ```

   `log_dir` 为日志文件夹，TensorBoard 会从此目录加载数据。

2. **记录训练损失和准确率**：

   ```python
   writer.add_scalar('Loss/train', epoch_loss, epoch)
   writer.add_scalar('Accuracy/train', epoch_acc, epoch)
   ```

3. **记录混淆矩阵热力图**：

   ```python
   cm_image = plot_confusion_matrix(cm)  # 用函数绘制混淆矩阵的热力图
   writer.add_image('Confusion Matrix Heatmap', cm_image, epoch)
   ```

4. **启动 TensorBoard**：

   ```bash
   tensorboard --logdir=runs
   ```

5. **在浏览器中访问**：
   打开浏览器，访问 `http://localhost:6006/`，即可看到各种可视化内容。

##### 4.3 **TensorBoard 可视化内容**

* **Scalars**：显示训练过程中的损失、准确率等曲线。
* **Images**：显示混淆矩阵的热力图。
* **PR Curves**：显示模型的精度召回曲线。

---

#### **五、总结**

* **混淆矩阵**是评估分类模型性能的基础工具，可以帮助分析错误样本并进一步优化模型。
* **性能度量**（如准确率、精确率、召回率、F1 分数）可以从混淆矩阵中计算得出，帮助全面评估模型。
* **错误样本分析**有助于理解模型的弱点，从而进行有针对性的改进。
* **TensorBoard**是一个强大的可视化工具，它可以帮助你实时监控训练过程，查看模型的训练曲线、混淆矩阵等。


In [4]:
# Import necessary libraries
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
import os
from kaggle.api.kaggle_api_extended import KaggleApi
from torchvision import datasets, transforms, models
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from torch.utils.tensorboard import SummaryWriter
import seaborn as sns

In [5]:
# 数据预处理：包括归一化、调整大小、数据增强等
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet50需要输入大小为224x224
    transforms.RandomHorizontalFlip(),  # 数据增强：随机水平翻转
    transforms.ToTensor(),  # 转换为Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 归一化
])

# 加载数据集
train_dataset = datasets.ImageFolder(root='datasets/train', transform=transform)
test_dataset = datasets.ImageFolder(root='datasets/test', transform=transform)

# 划分训练集和验证集，比例为 80% 训练集，20% 验证集
train_size = int(0.8 * len(train_dataset))  # 80% 为训练集
val_size = len(train_dataset) - train_size  # 剩下的 20% 为验证集
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# 使用DataLoader加载数据，设置batch_size为32，shuffle为True
batch_size = 480   # 根据显存可调，32→64→128…，直到显存占满为止
num_workers = 4   # 用于 DataLoader 的并行加载，可按 CPU 核数调节

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [6]:
# 加载预训练的 ResNet50 模型
model = models.resnet50(pretrained=True)

# 修改最后一层，适应二分类任务（输出层为2类）
model.fc = nn.Linear(model.fc.in_features, 2)

# 将模型移到GPU（如果可用）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

d:\miniconda3\envs\torch_cu124\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\miniconda3\envs\torch_cu124\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# 冻结 ResNet50 的卷积层，仅训练全连接层
for param in model.parameters():
    param.requires_grad = False

# 仅解冻全连接层
for param in model.fc.parameters():
    param.requires_grad = True

# 使用交叉熵损失函数
criterion = nn.CrossEntropyLoss()

# 使用 AdamW 优化器来加速训练
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 如果使用学习率调度器
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

#### 创建混淆矩阵的函数定义
---

In [15]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from io import BytesIO
from PIL import Image

def plot_confusion_matrix(cm, class_names=None):
    # 创建一个混淆矩阵的热力图
    plt.figure(figsize=(8, 6))
    
    if class_names is not None:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    else:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')

    # 使用 BytesIO 在内存中保存图像
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    
    # 使用 PIL 读取图像并转换为 numpy 数组
    img = Image.open(buf)
    img_array = np.array(img)

    # 将 numpy 数组转换为 PyTorch Tensor
    img_tensor = torch.from_numpy(img_array).permute(2, 0, 1)  # 转换为 C x H x W 格式

    plt.close()  # 关闭当前图像
    return img_tensor


#### 创建训练与评估的函数

---

In [1]:
# 用于可视化的数组
train_acc = []
train_loss = []
test_acc = []
test_loss = []

# ======= 定义训练和测试函数 =======
# 初始化 TensorBoard SummaryWriter
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        optimizer.step()

        # 统计损失
        running_loss += loss.item() * labels.size(0)
        
        # 统计准确率
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # 收集所有预测和标签，用于计算混淆矩阵
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # 更新进度条
        acc = 100. * correct / total
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{acc:.3f}%")

    # 计算每个epoch的平均损失和准确率
    epoch_loss = running_loss / total
    epoch_acc = correct / total * 100.

    # 计算混淆矩阵
    cm = confusion_matrix(all_labels, all_preds)

    # Plot confusion matrix and log it to TensorBoard
    cm_image = plot_confusion_matrix(cm)
    writer.add_image('Confusion Matrix Heatmap', cm_image, epoch)
    
    
    # 记录到 TensorBoard
    writer.add_scalar('Loss/train', epoch_loss, epoch)
    writer.add_scalar('Accuracy/train', epoch_acc, epoch)

    # 记录混淆矩阵
    writer.add_pr_curve('Confusion Matrix', torch.tensor(all_labels), torch.tensor(all_preds), global_step=epoch)

    # 记录混淆矩阵的热力图
    cm_image = plot_confusion_matrix(cm)
    writer.add_image('Confusion Matrix Heatmap', cm_image, epoch)

    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * labels.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total * 100.
    return epoch_loss, epoch_acc

In [16]:
# ======= 主训练循环（含早停） =======
best_val_acc = 0.0
patience = 5
trigger_times = 0
num_epochs = 20
writer = SummaryWriter(log_dir='runs/experiment_1')
for epoch in range(1, num_epochs+1):
    train_l, train_a = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    val_l, val_a     = evaluate(model, val_loader, criterion)

    
    # 更新学习率
    scheduler.step()    

    train_loss.append(train_l)
    train_acc.append(train_a)
    test_loss.append(val_l)
    test_acc.append(val_a)

    print(f"\nEpoch: {epoch}/{num_epochs}")
    print(f"Train Loss: {train_l:.3f} | Train Acc: {train_a:.2f}%")
    print(f"Test Loss:  {val_l:.3f} | Test Acc:  {val_a:.2f}%\n")

    # 早停逻辑
    if val_a > best_val_acc:
        best_val_acc = val_a
        trigger_times = 0
        # 保存最佳模型
        torch.save(model.state_dict(), 'best_resnet50.pth')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered")
            break

writer.close()


Training:   0%|          | 0/41 [00:00<?, ?it/s]


Epoch: 1/20
Train Loss: 0.017 | Train Acc: 99.67%
Test Loss:  0.016 | Test Acc:  99.71%




Epoch: 2/20
Train Loss: 0.015 | Train Acc: 99.75%
Test Loss:  0.015 | Test Acc:  99.73%




Epoch: 3/20
Train Loss: 0.016 | Train Acc: 99.70%
Test Loss:  0.016 | Test Acc:  99.69%




Epoch: 4/20
Train Loss: 0.016 | Train Acc: 99.72%
Test Loss:  0.015 | Test Acc:  99.71%



...


Epoch: 8/20
Train Loss: 0.015 | Train Acc: 99.73%
Test Loss:  0.016 | Test Acc:  99.73%




Epoch: 9/20
Train Loss: 0.015 | Train Acc: 99.68%
Test Loss:  0.016 | Test Acc:  99.71%




Epoch: 10/20
Train Loss: 0.015 | Train Acc: 99.70%
Test Loss:  0.016 | Test Acc:  99.71%




Epoch: 11/20
Train Loss: 0.015 | Train Acc: 99.73%
Test Loss:  0.015 | Test Acc:  99.73%




Epoch: 12/20
Train Loss: 0.015 | Train Acc: 99.70%
Test Loss:  0.016 | Test Acc:  99.71%

Early stopping triggered
